# Feature Importance Analysis & Multi-Model Tuning

This notebook walks through four phases:
1. **Feature importance** — rank all 13 churn features across 5 methods (XGBoost gain/weight, SHAP, permutation importance, mutual information)
2. **Feature selection** — measure AUC-ROC for top-K subsets to find the minimal effective feature set
3. **Multi-model Optuna tuning** — tune XGBoost, LightGBM, and RandomForest with 25 trials each
4. **Model comparison** — pick the winner and save it

Data is loaded via `src.churn.load_and_preprocess()` to guarantee the same 13 features and identical train/test split used in the original churn model.

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import mlflow
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from IPython.display import Image

from src.churn import load_and_preprocess
from src.model_tuner import (
    compute_feature_importance,
    feature_selection_experiment,
    plot_feature_importance_heatmap,
    plot_auc_vs_feature_count,
    plot_model_comparison,
    _tune_xgboost,
    _tune_lgbm,
    _tune_rf,
    _evaluate,
    FIGURES_DIR,
    DATA_PROCESSED,
    MODELS_DIR,
)

%matplotlib inline

## 1. Load Data

`load_and_preprocess()` builds the 13 behavioral features from `retail_clean.csv` and attaches churn labels from `rfm_scores.csv`. Using the same function as the original churn model guarantees an identical feature set.

In [ ]:
X, y, cids = load_and_preprocess()
print(f"Samples: {X.shape[0]}  |  Features: {X.shape[1]}  |  Churn rate: {y.mean():.3f}")
print("Features:", X.columns.tolist())

In [ ]:
X_tr, X_te, y_tr, y_te, _, _ = train_test_split(
    X.values, y.values, cids.values,
    test_size=0.20, stratify=y.values, random_state=42,
)
X_tr_df = pd.DataFrame(X_tr, columns=X.columns)
X_te_df = pd.DataFrame(X_te, columns=X.columns)
print(f"Train: {len(X_tr)}  |  Test: {len(X_te)}")

## 2. Feature Importance Analysis

We compute importance from 5 independent methods on the baseline XGBoost model (`models/churn_xgboost.pkl`):

| Method | What it measures |
|---|---|
| XGBoost Gain | Average loss reduction per split on this feature |
| XGBoost Weight | Number of times the feature is used to split |
| SHAP | Mean absolute SHAP value on held-out test set |
| Permutation | Drop in AUC when this feature is randomly shuffled |
| Mutual Info | Statistical dependency between feature and target |

In [ ]:
baseline      = joblib.load(MODELS_DIR / "churn_xgboost.pkl")
importance_df = compute_feature_importance(baseline, X_tr_df, y_tr, X_te_df, y_te)
importance_df.to_csv(DATA_PROCESSED / "feature_importance_ranks.csv", index=False)

rank_cols = [c for c in importance_df.columns if c.startswith("rank_")]
display(importance_df[["feature", *rank_cols, "mean_rank"]].to_string(index=False))

In [ ]:
plot_feature_importance_heatmap(importance_df, FIGURES_DIR / "feature_importance_comparison.png")
Image(str(FIGURES_DIR / "feature_importance_comparison.png"))

## 3. Feature Selection Experiment

We retrain XGBoost with fixed hyperparameters using only the top-K features (ranked by mean rank above). This reveals whether the model can be simplified without a meaningful AUC loss.

In [ ]:
ranked_features = importance_df["feature"].tolist()
spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
base_params = {
    "max_depth": 5, "learning_rate": 0.1, "n_estimators": 200,
    "subsample": 0.8, "colsample_bytree": 0.8, "scale_pos_weight": spw,
}

k_results = feature_selection_experiment(
    X_tr, y_tr, X_te, y_te, X.columns.tolist(), ranked_features, base_params
)
for k, auc_val in sorted(k_results.items()):
    print(f"  Top-{k:2d} features: CV AUC = {auc_val:.4f}")

In [ ]:
plot_auc_vs_feature_count(k_results, FIGURES_DIR / "auc_vs_feature_count.png")
Image(str(FIGURES_DIR / "auc_vs_feature_count.png"))

## 4. Multi-Model Optuna Tuning

Each model is tuned independently with 25 Optuna trials (TPE sampler, 5-fold CV, maximising AUC-ROC). Results are logged to the `model_tuning` MLflow experiment.

> For the production run with 75 trials use: `python run_model_tuner.py`

In [ ]:
N_TRIALS = 25  # reduced for notebook; run_model_tuner.py uses 75
mlflow.set_experiment("model_tuning")

In [ ]:
print(f"Tuning XGBoost ({N_TRIALS} trials) ...")
xgb_params, xgb_cv_auc = _tune_xgboost(X_tr, y_tr, n_trials=N_TRIALS)
xgb_model = XGBClassifier(**xgb_params, tree_method="hist", random_state=42, n_jobs=-1, verbosity=0)
xgb_model.fit(X_tr, y_tr)
xgb_metrics = _evaluate(xgb_model, X_te, y_te)
print(f"  AUC-ROC: {xgb_metrics['auc_roc']:.4f}  F1: {xgb_metrics['f1']:.4f}  P@20: {xgb_metrics['precision_top20']:.4f}")

with mlflow.start_run(run_name="xgb_tuned"):
    mlflow.log_params({**xgb_params, "n_trials": N_TRIALS, "model": "XGBoost"})
    mlflow.log_metrics({**xgb_metrics, "cv_auc": xgb_cv_auc})
    mlflow.set_tag("model_type", "XGBoost")

In [ ]:
print(f"Tuning LightGBM ({N_TRIALS} trials) ...")
lgbm_params, lgbm_cv_auc = _tune_lgbm(X_tr, y_tr, n_trials=N_TRIALS)
lgbm_model = lgb.LGBMClassifier(**lgbm_params, random_state=42, n_jobs=-1, verbose=-1)
lgbm_model.fit(X_tr, y_tr)
lgbm_metrics = _evaluate(lgbm_model, X_te, y_te)
print(f"  AUC-ROC: {lgbm_metrics['auc_roc']:.4f}  F1: {lgbm_metrics['f1']:.4f}  P@20: {lgbm_metrics['precision_top20']:.4f}")

with mlflow.start_run(run_name="lgbm_tuned"):
    mlflow.log_params({**lgbm_params, "n_trials": N_TRIALS, "model": "LightGBM"})
    mlflow.log_metrics({**lgbm_metrics, "cv_auc": lgbm_cv_auc})
    mlflow.set_tag("model_type", "LightGBM")

In [ ]:
print(f"Tuning RandomForest ({N_TRIALS} trials) ...")
rf_params, rf_cv_auc = _tune_rf(X_tr, y_tr, n_trials=N_TRIALS)
rf_model = RandomForestClassifier(**rf_params, class_weight="balanced", random_state=42, n_jobs=-1)
rf_model.fit(X_tr, y_tr)
rf_metrics = _evaluate(rf_model, X_te, y_te)
print(f"  AUC-ROC: {rf_metrics['auc_roc']:.4f}  F1: {rf_metrics['f1']:.4f}  P@20: {rf_metrics['precision_top20']:.4f}")

with mlflow.start_run(run_name="rf_tuned"):
    mlflow.log_params({**rf_params, "n_trials": N_TRIALS, "model": "RandomForest"})
    mlflow.log_metrics({**rf_metrics, "cv_auc": rf_cv_auc})
    mlflow.set_tag("model_type", "RandomForest")

## 5. Model Comparison & Best Model

In [ ]:
comparison    = {"XGBoost": xgb_metrics, "LightGBM": lgbm_metrics, "RandomForest": rf_metrics}
tuned_models  = {"XGBoost": xgb_model,   "LightGBM": lgbm_model,   "RandomForest": rf_model}

print(f"  {'Model':<15} {'AUC-ROC':>8}  {'F1':>7}  {'P@Top20':>9}")
print("  " + "-" * 44)
for name, m in comparison.items():
    print(f"  {name:<15} {m['auc_roc']:>8.4f}  {m['f1']:>7.4f}  {m['precision_top20']:>9.4f}")

best_name  = max(comparison, key=lambda m: comparison[m]["auc_roc"])
best_model = tuned_models[best_name]
print(f"\nBest model: {best_name}  (AUC-ROC: {comparison[best_name]['auc_roc']:.4f})")

In [ ]:
joblib.dump(best_model, MODELS_DIR / "best_churn_model.pkl")
plot_model_comparison(comparison, FIGURES_DIR / "model_comparison.png")

with mlflow.start_run(run_name="best_model"):
    mlflow.log_params({"model_type": best_name, "n_trials": N_TRIALS})
    mlflow.log_metrics(comparison[best_name])
    mlflow.set_tag("best_model", "True")
    mlflow.set_tag("winner", best_name)
    mlflow.log_artifact(str(MODELS_DIR / "best_churn_model.pkl"))
    for png in ["feature_importance_comparison.png", "auc_vs_feature_count.png", "model_comparison.png"]:
        mlflow.log_artifact(str(FIGURES_DIR / png))

Image(str(FIGURES_DIR / "model_comparison.png"))